In [21]:
import io, os, sys, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Image
from pathlib import Path
from scipy.interpolate import interp1d
from scipy.signal import lfilter, butter, sosfiltfilt
import torch

warnings.filterwarnings("ignore", message=".*NumPy.*")

PROJECT_ROOT = Path("/home/metamobility3/Jinwoo/os_kinetics")
sys.path.insert(0, str(PROJECT_ROOT))
from dataset import IK_DOF_NAMES, MOMENT_NAMES, _compute_velocity
from model import TCN

## 1 · Configuration

In [ ]:
OPENSIM_ROOT   = Path("/home/metamobility3/Jinwoo/AB03_Ilseung/")
CHECKPOINT     = PROJECT_ROOT / "runs/0510_ik_id_no_stair_epic_only/best_model.pt"

SUBJECT_MASS_KG    = 84.4      # model output (N·m/kg) × mass → N·m for comparison
ID_LPF_CUTOFF_HZ   = 6.0       # must match out_lpf_hz in controller YAML
ID_LPF_ORDER       = 4         # cascaded 1st-order EMA stages (causal)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ── model output channel order (after unilateral-paired bilateral inference) ──
#   Right side first, then Left; each side: hip_flexion, knee_angle, ankle_angle
OUTPUT_LABELS = [
    ("hip_flexion_r",  "Hip Flexion R"),
    ("knee_angle_r",   "Knee Angle R"),
    ("ankle_angle_r",  "Ankle Angle R"),
    ("hip_flexion_l",  "Hip Flexion L"),
    ("knee_angle_l",   "Knee Angle L"),
    ("ankle_angle_l",  "Ankle Angle L"),
]
N_OUT = len(OUTPUT_LABELS)

# Corresponding OpenSim ID .sto column names (N·m, not normalised by mass)
ID_COLS = [
    "hip_flexion_r_moment",
    "knee_angle_r_moment",
    "ankle_angle_r_moment",
    "hip_flexion_l_moment",
    "knee_angle_l_moment",
    "ankle_angle_l_moment",
]

# ── colour palette ─────────────────────────────────────────────────────────────
PALETTE = {
    "id":       "#2196F3",
    "model":    "#F44336",
    "residual": "#9C27B0",
}
# Per-channel colours (alternating warm/cool for R/L)
CHAN_COLORS_MODEL = ["#E53935","#D81B60","#8E24AA","#1E88E5","#039BE5","#00ACC1"]
CHAN_COLORS_ID    = ["#90CAF9","#CE93D8","#A5D6A7","#FFCC80","#B0BEC5","#80CBC4"]

# ── exo motor torque subtracted from OpenSim ID (same as compare_opensim_id_vs_model) ──
#   τ_exo [N·m] = pred_nm × scale  on the aligned time grid  (pred_nm = nmpkg × mass, causal LPF applied).
EXO_TORQUE_SCALE = {
    "hip-exo":  0.12,
    "knee-exo": 0.09,
}
# Output channel order: 0,3 = hip R/L; 1,4 = knee R/L; 2,5 = ankle R/L — edit per hardware if needed.
EXO_SPEC_AWINDA_HIP_KNEE = [
    ("hip-exo",  (0, 3)),
    ("knee-exo", (1, 4)),
]

# ── trial mapping: stem → (exo_subfolder, opensim_condition_stem[, exo_spec]) ─
#   exo_spec: None = no τ_exo subtraction; else list of (EXO_TORQUE_SCALE key, channel indices).
TRIALS = {
    "awinda_LG_0p8mps": ("awinda", "LG_0p8mps", EXO_SPEC_AWINDA_HIP_KNEE),
    "awinda_LG_1p2mps": ("awinda", "LG_1p2mps", EXO_SPEC_AWINDA_HIP_KNEE),
    "awinda_LG_1p6mps": ("awinda", "LG_1p6mps", EXO_SPEC_AWINDA_HIP_KNEE),
    "awinda_RA_0p8mps": ("awinda", "RA_0p8mps", EXO_SPEC_AWINDA_HIP_KNEE),
    "awinda_RD_0p8mps": ("awinda", "RD_0p8mps", EXO_SPEC_AWINDA_HIP_KNEE),
}

print("Trial manifest:")
for stem, spec in TRIALS.items():
    exo, cond = spec[0], spec[1]
    ik_ok  = (OPENSIM_ROOT / exo / "ik" / f"{cond}_ik.mot").exists()
    id_ok  = (OPENSIM_ROOT / exo / "id" / f"{cond}_id.sto").exists()
    print(f"  {'✓' if ik_ok else '✗'} ik  {'✓' if id_ok else '✗'} id   {stem}")
print(f"\nCheckpoint: {'✓' if CHECKPOINT.exists() else '✗'}  {CHECKPOINT}")

Trial manifest:
  ✓ ik  ✓ id   awinda_LG_0p8mps
  ✓ ik  ✓ id   awinda_LG_1p2mps
  ✓ ik  ✓ id   awinda_LG_1p6mps
  ✓ ik  ✓ id   awinda_RA_0p8mps
  ✓ ik  ✓ id   awinda_RD_0p8mps

Checkpoint: ✓  /home/metamobility3/Jinwoo/os_kinetics/runs/0510_ik_id_no_stair_epic_only/best_model.pt


## 2 · Helper functions

In [23]:
def parse_opensim_file(path: Path) -> pd.DataFrame:
    """Read an OpenSim .sto or .mot file; returns a DataFrame indexed by time."""
    with open(path) as fh:
        for i, line in enumerate(fh):
            if line.strip() == "endheader":
                header_end = i
                break
    df = pd.read_csv(path, sep=r"\s+", skiprows=header_end + 1)
    return df.set_index("time")


def lpf_id(signal: np.ndarray, fs: float,
           cutoff_hz: float = ID_LPF_CUTOFF_HZ,
           order: int = ID_LPF_ORDER) -> np.ndarray:
    """
    Causal cascaded EMA low-pass — replicates _CausalLowPass from cascade.py.
    alpha = dt / (tau + dt),  tau = 1/(2π·cutoff_hz).
    Applied `order` times in series; initialised to first sample.
    """
    dt    = 1.0 / float(fs)
    tau   = 1.0 / (2.0 * np.pi * float(cutoff_hz))
    alpha = dt / (tau + dt)
    b = np.array([alpha])
    a = np.array([1.0, -(1.0 - alpha)])
    y = signal.astype(float).copy()
    for _ in range(order):
        zi = np.array([y[0] * (1.0 - alpha)])
        y, _ = lfilter(b, a, y, zi=zi)
    return y.astype(signal.dtype)


def lpf_id_multichannel(signals: np.ndarray, fs: float) -> np.ndarray:
    """Apply lpf_id independently to each column of an (T, C) array."""
    return np.stack([lpf_id(signals[:, c], fs) for c in range(signals.shape[1])], axis=1)


def lowpass_zero_phase(
    data: np.ndarray,
    time: np.ndarray,
    cutoff_hz: float = None,
    order: int = None,
) -> np.ndarray:
    """
    Zero-phase forward-backward Butterworth LPF (sosfiltfilt).
    Matches _lowpass_zero_phase in dataset.py — used for IK position denoising
    and velocity smoothing during training/evaluation.
    Defaults to IK_LPF_CUTOFF_HZ / IK_LPF_ORDER read from the checkpoint config.
    """
    _cut = cutoff_hz if cutoff_hz is not None else IK_LPF_CUTOFF_HZ
    _ord = order     if order     is not None else IK_LPF_ORDER
    fs = 1.0 / float(np.median(np.diff(time.astype(np.float64))))
    nyquist = 0.5 * fs
    if _cut <= 0 or _cut >= nyquist or data.shape[0] < 8:
        return data
    sos = butter(_ord, _cut / nyquist, btype="low", output="sos")
    return sosfiltfilt(sos, data.astype(np.float64), axis=0)


print("Helpers defined.")

Helpers defined.


## 3 · Load model checkpoint

In [24]:
ckpt = torch.load(str(CHECKPOINT), map_location=DEVICE, weights_only=False)

cfg = ckpt["model_config"]
model = TCN(
    n_input_channels=cfg["n_input_channels"],
    n_output_channels=cfg["n_output_channels"],
    hidden_channels=cfg["hidden_channels"],
    n_blocks=cfg["n_blocks"],
    kernel_size=cfg["kernel_size"],
    dropout=cfg["dropout"],
).to(DEVICE)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

WINDOW_SIZE     = int(ckpt["window_size"])
INPUT_INDICES   = list(ckpt["input_indices"])    # [6,9,10,13,16,17]
MOMENT_INDICES  = list(ckpt["moment_indices"])   # [3,12,14,6,13,15]
DOF_NAMES       = list(ckpt["dof_names"])        # ['hip_flexion','knee_angle','ankle_angle']
# ROLLOUT_STEP    = int(ckpt.get("rollout_decimate_step", 1))   # 2 → ~200 Hz → ~100 Hz
ROLLOUT_STEP    = 1

# Read IK/velocity preprocessing LPF params from checkpoint config
# (same as ID_LPF_CUTOFF_HZ / ID_LPF_ORDER for this run, but kept explicit)
_cfg_json_path = CHECKPOINT.parent / "config.json"
if _cfg_json_path.exists():
    import json as _json
    _cfg_json = _json.load(open(_cfg_json_path))
    IK_LPF_CUTOFF_HZ = float(_cfg_json.get("lowpass_cutoff_hz", 6.0))
    IK_LPF_ORDER     = int(_cfg_json.get("lowpass_order", 4))
else:
    IK_LPF_CUTOFF_HZ = 6.0
    IK_LPF_ORDER     = 4

# Stats stored for diagnostic/OOD reference only — NOT used at inference
# (checkpoint was trained with normalize=False; model receives raw radians/rad·s⁻¹)
norm     = ckpt["normalization"]
POS_MEAN = np.array(norm["pos_mean"])   # shape (23,)
POS_STD  = np.array(norm["pos_std"])
VEL_MEAN = np.array(norm["vel_mean"])
VEL_STD  = np.array(norm["vel_std"])

# Split into R / L halves for unilateral-paired inference
H = len(INPUT_INDICES) // 2
INPUT_IDX_R  = INPUT_INDICES[:H]   # [6,9,10]  → hip_r, knee_r, ankle_r
INPUT_IDX_L  = INPUT_INDICES[H:]   # [13,16,17] → hip_l, knee_l, ankle_l

print(f"Model: {cfg['n_input_channels']}→{cfg['n_output_channels']} channels, "
      f"window={WINDOW_SIZE}, blocks={cfg['n_blocks']}, hidden={cfg['hidden_channels']}")
print(f"Rollout step: {ROLLOUT_STEP}  (native ~200 Hz → ~{200//ROLLOUT_STEP} Hz)")
print(f"normalize=False — inputs fed raw (radians / rad·s⁻¹), no z-scoring")
print(f"Input DoFs R : {[IK_DOF_NAMES[i] for i in INPUT_IDX_R]}")
print(f"Input DoFs L : {[IK_DOF_NAMES[i] for i in INPUT_IDX_L]}")
print(f"Output moments: {[MOMENT_NAMES[i] for i in MOMENT_INDICES]}")
print(f"Device: {DEVICE}")

Model: 6→3 channels, window=100, blocks=5, hidden=80
Rollout step: 1  (native ~200 Hz → ~200 Hz)
normalize=False — inputs fed raw (radians / rad·s⁻¹), no z-scoring
Input DoFs R : ['hip_flexion_r', 'knee_angle_r', 'ankle_angle_r']
Input DoFs L : ['hip_flexion_l', 'knee_angle_l', 'ankle_angle_l']
Output moments: ['hip_flexion_r', 'knee_angle_r', 'ankle_angle_r', 'hip_flexion_l', 'knee_angle_l', 'ankle_angle_l']
Device: cuda


## 4 · Inference functions

In [25]:
@torch.no_grad()
def causal_infer_unilateral(pos_in: np.ndarray, vel_in: np.ndarray) -> np.ndarray:
    """
    Run causal TCN inference for one side (R or L).
    pos_in, vel_in : (T, 3) raw position (rad) and velocity (rad/s) — NOT normalised.
    The checkpoint was trained with normalize=False; the model expects physical-unit inputs.
    Returns predictions shape (T, 3) in N·m/kg.
    """
    x = np.concatenate([pos_in, vel_in], axis=1).astype(np.float32)  # (T, 6)

    n    = x.shape[0]
    W    = WINDOW_SIZE
    n_out = cfg["n_output_channels"]
    pred = np.zeros((n, n_out), dtype=np.float64)

    def _fwd(start):
        # np.ascontiguousarray ensures the CUDA kernel gets a C-contiguous buffer
        arr = np.ascontiguousarray(x[start:start + W].T)          # (6, W) C-contiguous
        xw  = torch.from_numpy(arr).unsqueeze(0).to(DEVICE)       # (1, 6, W)
        return model(xw).squeeze(0).detach().cpu().numpy().T       # (W, n_out)

    # First window: fill bootstrap predictions for t = 0 .. W-1
    pw0 = _fwd(0)
    for g in range(W):
        pred[g] = pw0[g]

    # Sliding causal windows: update only the newest timestep per shift
    for start in range(1, n - W + 1):
        pred[start + W - 1] = _fwd(start)[W - 1]

    return pred.astype(np.float32)   # (T, 3)


def run_inference(pos_all: np.ndarray, vel_all: np.ndarray) -> np.ndarray:
    """
    Bilateral inference via unilateral-paired strategy.
    pos_all, vel_all : (T, 23) — full IK position/velocity arrays (radians, rad/s).
    Returns pred : (T, 6)  in N·m/kg, order: [hip_r, knee_r, ankle_r, hip_l, knee_l, ankle_l].
    """
    pred_r = causal_infer_unilateral(pos_all[:, INPUT_IDX_R], vel_all[:, INPUT_IDX_R])
    pred_l = causal_infer_unilateral(pos_all[:, INPUT_IDX_L], vel_all[:, INPUT_IDX_L])
    return np.concatenate([pred_r, pred_l], axis=1)   # (T, 6)


print("Inference functions defined.")

Inference functions defined.


## 5 · Pre-load all trials

Reads IK, runs inference, reads ID, and applies the 6 Hz causal LPF to ID moments. On the aligned grid, **applied exo torque** is subtracted from gross ID the same way as `compare_opensim_id_vs_model.ipynb`: `τ_exo = pred_nm × EXO_TORQUE_SCALE[…]` per configured output channel, then **ID_net = ID_gross − τ_exo** for metrics and plots. IK and ID share the same OpenSim time axis (no GPIO sync).

In [26]:
TRIAL_DATA = {}


def _exo_torque_nm(pred_nm_aligned: np.ndarray, exo_spec) -> np.ndarray:
    """τ_exo [N·m] = pred_nm × EXO_TORQUE_SCALE[key] per channel (compare_opensim_id_vs_model)."""
    tau = np.zeros_like(pred_nm_aligned, dtype=np.float64)
    if not exo_spec:
        return tau
    for scale_key, chans in exo_spec:
        sc = float(EXO_TORQUE_SCALE[scale_key])
        for c in chans:
            tau[:, int(c)] += pred_nm_aligned[:, int(c)] * sc
    return tau


for stem, spec in TRIALS.items():
    if len(spec) == 2:
        exo, cond = spec
        exo_spec = None
    else:
        exo, cond, exo_spec = spec
    ik_path = OPENSIM_ROOT / exo / "ik" / f"{cond}_ik.mot"
    id_path = OPENSIM_ROOT / exo / "id" / f"{cond}_id.sto"

    if not ik_path.exists() or not id_path.exists():
        print(f"[SKIP] {stem} — missing file(s)")
        continue

    print(f"Processing {stem} …", end=" ", flush=True)

    # ── IK: load, degrees → radians, compute velocity ─────────────────────────
    ik_df   = parse_opensim_file(ik_path)
    t_ik    = ik_df.index.to_numpy(float)
    # Columns may not include all IK_DOF_NAMES; fill missing with 0
    pos_deg = np.zeros((len(t_ik), len(IK_DOF_NAMES)), dtype=np.float64)
    for j, name in enumerate(IK_DOF_NAMES):
        if name in ik_df.columns:
            pos_deg[:, j] = ik_df[name].to_numpy(float)

    # Decimate only if native rate is ~200 Hz (AddBiomechanics); skip if already ~100 Hz
    ik_fs_native = 1.0 / float(np.median(np.diff(t_ik)))
    effective_step = ROLLOUT_STEP if ik_fs_native > 150.0 else 1
    if effective_step > 1:
        t_ik    = t_ik[::effective_step]
        pos_deg = pos_deg[::effective_step]

    pos_rad = np.deg2rad(pos_deg)                       # (T, 23) radians
    pos_rad = lowpass_zero_phase(pos_rad, t_ik)         # zero-phase 6 Hz 4-ord (matches dataset LPF on positions)
    vel_rad = _compute_velocity(pos_rad, t_ik)          # (T, 23) rad/s — same hybrid B-spline/rotation method
    vel_rad = lowpass_zero_phase(vel_rad, t_ik)         # zero-phase 6 Hz 4-ord (matches dataset velocity LPF)

    ik_fs      = 1.0 / float(np.median(np.diff(t_ik)))

    # ── Run bilateral causal inference ────────────────────────────────────────
    pred_nmpkg = run_inference(pos_rad, vel_rad)   # (T, 6)  N·m/kg
    pred_nm    = pred_nmpkg * SUBJECT_MASS_KG      # (T, 6)  N·m
    pred_nm    = lpf_id_multichannel(pred_nm, fs=ik_fs)   # causal 6 Hz 4-ord LPF (matches controller)

    # ── ID: load, apply 6 Hz causal LPF ──────────────────────────────────────
    id_df      = parse_opensim_file(id_path)
    t_id       = id_df.index.to_numpy(float)
    id_fs      = 1.0 / float(np.median(np.diff(t_id)))

    id_raw = np.zeros((len(t_id), N_OUT), dtype=np.float64)
    for c, col in enumerate(ID_COLS):
        if col in id_df.columns:
            id_raw[:, c] = id_df[col].to_numpy(float)
        else:
            id_raw[:, c] = np.nan
    id_filt = lpf_id_multichannel(id_raw, fs=id_fs)   # (T_id, 6)  N·m

    # ── Align IK and ID on their shared time axis ─────────────────────────────
    t_lo = max(t_ik[0],  t_id[0])
    t_hi = min(t_ik[-1], t_id[-1])
    mask = (t_ik >= t_lo) & (t_ik <= t_hi)
    t_common   = t_ik[mask]
    pred_common = pred_nm[mask]

    id_interp = np.stack([
        interp1d(t_id, id_filt[:, c], kind="linear",
                 bounds_error=False, fill_value=np.nan)(t_common)
        for c in range(N_OUT)
    ], axis=1)   # (T_common, 6) — gross ID (before subtracting applied exo torque)

    exo_torque = _exo_torque_nm(pred_common, exo_spec)
    id_net     = id_interp - exo_torque   # net ID for comparison (same as model notebook)

    # ── Per-channel metrics (pred vs ID − τ_exo) ───────────────────────────────
    metrics = []
    for c in range(N_OUT):
        v = ~np.isnan(id_net[:, c])
        if v.sum() > 1:
            rmse = float(np.sqrt(np.mean((pred_common[v, c] - id_net[v, c])**2)))
            r2   = float(np.corrcoef(pred_common[v, c], id_net[v, c])[0, 1])**2
        else:
            rmse, r2 = np.nan, np.nan
        metrics.append({"rmse": rmse, "r2": r2})

    TRIAL_DATA[stem] = {
        "stem":            stem,
        "exo":             exo,
        "cond":            cond,
        "exo_spec":        exo_spec,
        "t":               t_common,
        "t_rel":           t_common - t_common[0],
        "pred_nm":         pred_common,      # (T, 6)
        "id_nm_gross":     id_interp.copy(), # (T, 6) OpenSim ID only
        "exo_torque_nm":   exo_torque,       # (T, 6) τ_exo = pred × scale per spec
        "id_nm":           id_net,          # (T, 6) ID − τ_exo (comparison target)
        "metrics":         metrics,
    }

    mean_r2   = float(np.nanmean([m["r2"]   for m in metrics]))
    mean_rmse = float(np.nanmean([m["rmse"] for m in metrics]))
    dur = float(t_common[-1] - t_common[0])
    print(f"ok  ({dur:.1f} s, mean R²={mean_r2:.3f}, mean RMSE={mean_rmse:.2f} N·m)")

print(f"\nLoaded {len(TRIAL_DATA)} / {len(TRIALS)} trials.")

Processing awinda_LG_0p8mps … ok  (80.0 s, mean R²=0.853, mean RMSE=10.47 N·m)
Processing awinda_LG_1p2mps … ok  (80.0 s, mean R²=0.914, mean RMSE=9.29 N·m)
Processing awinda_LG_1p6mps … ok  (80.0 s, mean R²=0.943, mean RMSE=9.72 N·m)
Processing awinda_RA_0p8mps … ok  (80.0 s, mean R²=0.914, mean RMSE=11.04 N·m)
Processing awinda_RD_0p8mps … ok  (80.0 s, mean R²=0.888, mean RMSE=9.14 N·m)

Loaded 5 / 5 trials.


In [27]:
## 5b · Inspect model inputs (interactive)
# Per-trial cache so IK is only reloaded when the trial changes.
_INPUT_CACHE: dict = {}

def _load_input_data(stem: str) -> dict:
    if stem in _INPUT_CACHE:
        return _INPUT_CACHE[stem]
    exo  = TRIAL_DATA[stem]["exo"]
    cond = TRIAL_DATA[stem]["cond"]
    ik_df  = parse_opensim_file(OPENSIM_ROOT / exo / "ik" / f"{cond}_ik.mot")
    t      = ik_df.index.to_numpy(float)
    pos_deg = np.zeros((len(t), len(IK_DOF_NAMES)), dtype=np.float64)
    for j, name in enumerate(IK_DOF_NAMES):
        if name in ik_df.columns:
            pos_deg[:, j] = ik_df[name].to_numpy(float)
    t_full   = t.copy()
    pos_full = pos_deg.copy()

    # Decimate only if native rate is ~200 Hz (AddBiomechanics); skip if already ~100 Hz
    ik_fs_native = 1.0 / float(np.median(np.diff(t)))
    effective_step = ROLLOUT_STEP if ik_fs_native > 150.0 else 1
    if effective_step > 1:
        t       = t[::effective_step]
        pos_deg = pos_deg[::effective_step]
    pos_raw  = np.deg2rad(pos_deg)
    # GT IK: full-rate .mot angles (rad) linearly interpolated onto analysis times `t`
    rad_full = np.deg2rad(pos_full.astype(np.float64))
    pos_gt   = np.stack([
        interp1d(t_full, rad_full[:, j], kind="linear",
                 bounds_error=False, fill_value=np.nan)(t)
        for j in range(len(IK_DOF_NAMES))
    ], axis=1)
    vel_gt   = np.gradient(pos_gt, t.astype(np.float64), axis=0, edge_order=2)

    pos_filt = lowpass_zero_phase(pos_raw, t)
    vel_raw  = _compute_velocity(pos_filt, t)
    vel_filt = lowpass_zero_phase(vel_raw, t)
    # Training stats used for OOD reference only (normalize=False at inference)
    pos_norm_r = (pos_filt[:, INPUT_IDX_R] - POS_MEAN[INPUT_IDX_R]) / POS_STD[INPUT_IDX_R]
    vel_norm_r = (vel_filt[:, INPUT_IDX_R] - VEL_MEAN[INPUT_IDX_R]) / VEL_STD[INPUT_IDX_R]
    pos_norm_l = (pos_filt[:, INPUT_IDX_L] - POS_MEAN[INPUT_IDX_L]) / POS_STD[INPUT_IDX_L]
    vel_norm_l = (vel_filt[:, INPUT_IDX_L] - VEL_MEAN[INPUT_IDX_L]) / VEL_STD[INPUT_IDX_L]
    _INPUT_CACHE[stem] = dict(
        t=t, t_rel=t - t[0],
        fs=1.0 / float(np.median(np.diff(t))),
        pos_raw=pos_raw, pos_filt=pos_filt,
        vel_raw=vel_raw, vel_filt=vel_filt,
        pos_gt=pos_gt, vel_gt=vel_gt,
        pos_norm_r=pos_norm_r, vel_norm_r=vel_norm_r,
        pos_norm_l=pos_norm_l, vel_norm_l=vel_norm_l,
    )
    return _INPUT_CACHE[stem]

# ── Widgets ────────────────────────────────────────────────────────────────────
_first_stem_inp = next(iter(TRIAL_DATA))
_first_inp      = _load_input_data(_first_stem_inp)

inp_trial_dropdown = widgets.Dropdown(
    options=list(TRIAL_DATA.keys()),
    value=_first_stem_inp,
    description="Trial:",
    layout=widgets.Layout(width="500px"),
    style={"description_width": "80px"},
)
inp_time_slider = widgets.FloatRangeSlider(
    value=[_first_inp["t_rel"][0], _first_inp["t_rel"][-1]],
    min=_first_inp["t_rel"][0],
    max=_first_inp["t_rel"][-1],
    step=0.5,
    description="Time (s):",
    layout=widgets.Layout(width="700px"),
    style={"description_width": "80px"},
    continuous_update=False,
    readout_format=".1f",
)
inp_out = widgets.Output()

# DOF short labels for both sides
_dof_short = ["Hip Flexion", "Knee Angle", "Ankle Angle"]
_all_idx   = [INPUT_IDX_R, INPUT_IDX_L]
_colors_inp = {
    "raw_r":  "#90CAF9", "filt_r": "#1E88E5",   # blue family — Right
    "raw_l":  "#FFCC80", "filt_l": "#E65100",    # orange family — Left
    "gt_r":   "#212121", "gt_l":   "#2E7D32",    # dotted GT from full-rate .mot (interp → t)
    "norm_vel": "#9C27B0",                        # purple for vel z-score
}

# ── Draw ───────────────────────────────────────────────────────────────────────
def _draw_inputs(stem: str, t0: float, t1: float) -> None:
    d    = _load_input_data(stem)
    mask = (d["t_rel"] >= t0) & (d["t_rel"] <= t1)
    t    = d["t_rel"][mask]

    pos_raw  = d["pos_raw"][mask]
    pos_filt = d["pos_filt"][mask]
    pos_gt   = d["pos_gt"][mask]
    vel_raw  = d["vel_raw"][mask]
    vel_filt = d["vel_filt"][mask]
    vel_gt   = d["vel_gt"][mask]
    pos_norm = [d["pos_norm_r"][mask], d["pos_norm_l"][mask]]
    vel_norm = [d["vel_norm_r"][mask], d["vel_norm_l"][mask]]

    n_joints = len(_dof_short)
    fig, axes = plt.subplots(3, n_joints, figsize=(5.5 * n_joints, 9),
                             gridspec_kw={"hspace": 0.55, "wspace": 0.32})

    for j, joint in enumerate(_dof_short):
        idx_r, idx_l = INPUT_IDX_R[j], INPUT_IDX_L[j]

        # Row 0: position raw vs LPF, R and L overlaid
        ax = axes[0, j]
        ax.plot(t, pos_raw[:, idx_r],  color=_colors_inp["raw_r"],  lw=0.8, alpha=0.6)
        ax.plot(t, pos_filt[:, idx_r], color=_colors_inp["filt_r"], lw=1.6, label="R (LPF)")
        ax.plot(t, pos_raw[:, idx_l],  color=_colors_inp["raw_l"],  lw=0.8, alpha=0.6)
        ax.plot(t, pos_filt[:, idx_l], color=_colors_inp["filt_l"], lw=1.6, label="L (LPF)")
        ax.plot(t, pos_gt[:, idx_r],   color=_colors_inp["gt_r"],   lw=1.5, ls=":", label="R GT (IK .mot)")
        ax.plot(t, pos_gt[:, idx_l],   color=_colors_inp["gt_l"],   lw=1.5, ls=":", label="L GT (IK .mot)")
        ax.set_title(f"{joint}\nPosition (rad)", fontsize=9)
        ax.set_ylabel("rad", fontsize=8)
        ax.spines[["top", "right"]].set_visible(False)
        ax.legend(fontsize=6, loc="upper right", ncol=2)

        # Row 1: velocity raw vs LPF, R and L overlaid
        ax = axes[1, j]
        ax.plot(t, vel_raw[:, idx_r],  color=_colors_inp["raw_r"],  lw=0.8, alpha=0.6)
        ax.plot(t, vel_filt[:, idx_r], color=_colors_inp["filt_r"], lw=1.6, label="R (LPF)")
        ax.plot(t, vel_raw[:, idx_l],  color=_colors_inp["raw_l"],  lw=0.8, alpha=0.6)
        ax.plot(t, vel_filt[:, idx_l], color=_colors_inp["filt_l"], lw=1.6, label="L (LPF)")
        ax.plot(t, vel_gt[:, idx_r],   color=_colors_inp["gt_r"],   lw=1.4, ls=":", label="R GT (∂IK/∂t)")
        ax.plot(t, vel_gt[:, idx_l],   color=_colors_inp["gt_l"],   lw=1.4, ls=":", label="L GT (∂IK/∂t)")
        ax.set_title(f"{joint}\nVelocity (rad/s)", fontsize=9)
        ax.set_ylabel("rad/s", fontsize=8)
        ax.spines[["top", "right"]].set_visible(False)
        ax.legend(fontsize=6, loc="upper right", ncol=2)

        # Row 2: z-score reference (training dist — NOT applied at inference)
        ax = axes[2, j]
        ax.plot(t, pos_norm[0][:, j], color=_colors_inp["filt_r"], lw=1.3, label="pos R (z)")
        ax.plot(t, pos_norm[1][:, j], color=_colors_inp["filt_l"], lw=1.3, label="pos L (z)")
        ax.plot(t, vel_norm[0][:, j], color=_colors_inp["filt_r"], lw=1.1, ls="--", label="vel R (z)")
        ax.plot(t, vel_norm[1][:, j], color=_colors_inp["filt_l"], lw=1.1, ls="--", label="vel L (z)")
        ax.axhline( 3, color="gray", lw=0.7, ls=":", alpha=0.6)
        ax.axhline(-3, color="gray", lw=0.7, ls=":", alpha=0.6)
        ax.set_title(f"{joint}\nz-score (train dist ref — not applied at inference)", fontsize=8)
        ax.set_ylabel("σ", fontsize=8)
        ax.spines[["top", "right"]].set_visible(False)
        ax.legend(fontsize=6, loc="upper right", ncol=2)

    for ax in axes[-1, :]:
        ax.set_xlabel("Time (s)", fontsize=8)

    pos_ood_r = float(np.mean(np.abs(pos_norm[0]) > 3)) * 100
    vel_ood_r = float(np.mean(np.abs(vel_norm[0]) > 3)) * 100
    pos_ood_l = float(np.mean(np.abs(pos_norm[1]) > 3)) * 100
    vel_ood_l = float(np.mean(np.abs(vel_norm[1]) > 3)) * 100
    fig.suptitle(
        f"Model input diagnostics — {stem}  (fs ≈ {d['fs']:.0f} Hz)\n"
        f"Dotted GT = full-rate OpenSim IK (interp to grid); solid LPF = training-like chain.  "
        f"Outside ±3σ:  R pos={pos_ood_r:.1f}% vel={vel_ood_r:.1f}%   "
        f"L pos={pos_ood_l:.1f}% vel={vel_ood_l:.1f}%",
        fontsize=10, fontweight="bold",
    )

    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight", dpi=120)
    plt.close(fig)
    buf.seek(0)
    inp_out.clear_output(wait=True)
    with inp_out:
        display(Image(data=buf.read()))
        # Stats table — both sides
        print(f"\n{'DOF':<20} {'train μ_pos':>10} {'train σ_pos':>11}"
              f" {'data μ_pos':>10} {'data σ_pos':>11}"
              f"  {'train μ_vel':>10} {'train σ_vel':>11}"
              f" {'data μ_vel':>10} {'data σ_vel':>11}")
        print("-" * 118)
        for s, side in enumerate(["R", "L"]):
            for j, joint in enumerate(_dof_short):
                idx = _all_idx[s][j]
                pf  = pos_filt[:, idx]
                vf  = vel_filt[:, idx]
                print(f"  {joint+' '+side:<18}"
                      f"  {POS_MEAN[idx]:>10.4f}  {POS_STD[idx]:>11.4f}"
                      f"  {pf.mean():>10.4f}  {pf.std():>11.4f}"
                      f"    {VEL_MEAN[idx]:>10.4f}  {VEL_STD[idx]:>11.4f}"
                      f"  {vf.mean():>10.4f}  {vf.std():>11.4f}")

# ── Callbacks ──────────────────────────────────────────────────────────────────
def _on_inp_trial(change):
    d = _load_input_data(change["new"])
    inp_time_slider.min   = float(d["t_rel"][0])
    inp_time_slider.max   = float(d["t_rel"][-1])
    inp_time_slider.value = [float(d["t_rel"][0]), float(d["t_rel"][-1])]
    _redraw_inputs()

def _redraw_inputs(*_):
    _draw_inputs(inp_trial_dropdown.value, *inp_time_slider.value)

inp_trial_dropdown.observe(_on_inp_trial, names="value")
inp_time_slider.observe(_redraw_inputs,   names="value")

display(inp_trial_dropdown, inp_time_slider, inp_out)
_redraw_inputs()

Dropdown(description='Trial:', layout=Layout(width='500px'), options=('awinda_LG_0p8mps', 'awinda_LG_1p2mps', …

FloatRangeSlider(value=(0.0, 79.99), continuous_update=False, description='Time (s):', layout=Layout(width='70…

Output()

## 6 · Interactive comparison

In [28]:
_first_stem = next(iter(TRIAL_DATA))
_first_d    = TRIAL_DATA[_first_stem]

trial_dropdown = widgets.Dropdown(
    options=list(TRIAL_DATA.keys()),
    value=_first_stem,
    description="Trial:",
    layout=widgets.Layout(width="500px"),
    style={"description_width": "80px"},
)

time_slider = widgets.FloatRangeSlider(
    value=[_first_d["t_rel"][0], _first_d["t_rel"][-1]],
    min=_first_d["t_rel"][0],
    max=_first_d["t_rel"][-1],
    step=0.5,
    description="Time (s):",
    layout=widgets.Layout(width="700px"),
    style={"description_width": "80px"},
    continuous_update=False,
    readout_format=".1f",
)

show_residual = widgets.ToggleButton(
    value=False,
    description="Show residuals",
    button_style="",
    icon="signal",
    layout=widgets.Layout(width="160px"),
)

out = widgets.Output()


# ── draw ──────────────────────────────────────────────────────────────────────
def _draw(stem: str, t0: float, t1: float, residuals: bool) -> None:
    d    = TRIAL_DATA[stem]
    mask = (d["t_rel"] >= t0) & (d["t_rel"] <= t1)
    t    = d["t_rel"][mask]
    pred = d["pred_nm"][mask]   # (T_win, 6)
    id_m = d["id_nm"][mask]     # (T_win, 6)

    lpf_tag = f"{ID_LPF_CUTOFF_HZ:.0f} Hz LPF"
    n_rows   = N_OUT * 2 if residuals else N_OUT
    row_h    = 2.2
    fig_h    = n_rows * row_h + 0.6

    # 3×2 layout: rows = joints (hip/knee/ankle), cols = sides (R/L)
    n_joints = 3
    n_sides  = 2
    n_main   = n_joints * n_sides if not residuals else n_joints * n_sides * 2

    fig, axs = plt.subplots(
        n_joints * (2 if residuals else 1), n_sides,
        figsize=(16, fig_h),
        gridspec_kw={"hspace": 0.55, "wspace": 0.25},
        sharex=True,
    )
    if axs.ndim == 1:
        axs = axs.reshape(-1, n_sides)

    joint_names = ["Hip Flexion", "Knee Angle", "Ankle Angle"]
    side_names  = ["Right", "Left"]
    _exo_on     = bool(d.get("exo_spec"))
    _id_lbl     = (f"OpenSim ID − τ_exo ({lpf_tag})" if _exo_on
                  else f"OpenSim ID ({lpf_tag})")
    _res_title  = "Residual (Model − (ID − τ_exo))" if _exo_on else "Residual (Model − ID)"

    for j in range(n_joints):
        for s in range(n_sides):
            c_idx = j + s * n_joints   # channel: 0-2 = R, 3-5 = L
            label_key, label_str = OUTPUT_LABELS[c_idx]
            m   = metrics = d["metrics"][c_idx]

            # ── moment overlay row ───────────────────────────────────────────
            ax = axs[j * (2 if residuals else 1), s]
            ax.plot(t, id_m[:, c_idx],   color=PALETTE["id"],    lw=2.0,
                    label=_id_lbl)
            ax.plot(t, pred[:, c_idx],   color=PALETTE["model"], lw=1.8, ls="--",
                    label=f"Model (GT IK → TCN × {SUBJECT_MASS_KG:.0f} kg, {ID_LPF_CUTOFF_HZ:.0f} Hz LPF)")
            ax.set_ylabel("N·m", fontsize=10)
            ax.set_title(
                f"{joint_names[j]} — {side_names[s]}\n"
                f"RMSE={m['rmse']:.2f} N·m   R²={m['r2']:.3f}",
                fontsize=9,
            )
            ax.axhline(0, color="gray", lw=0.6, ls=":")
            ax.spines[["top","right"]].set_visible(False)
            ax.tick_params(labelsize=9)
            if j == 0 and s == 0:
                ax.legend(fontsize=8, loc="upper right")

            # ── residual row (optional) ──────────────────────────────────────
            if residuals:
                ax_r = axs[j * 2 + 1, s]
                resid = pred[:, c_idx] - id_m[:, c_idx]
                ax_r.plot(t, resid, color=PALETTE["residual"], lw=1.2)
                ax_r.axhline(0, color="black", lw=0.7, ls=":")
                ax_r.set_ylabel("N·m", fontsize=9)
                ax_r.set_title(_res_title, fontsize=8)
                ax_r.spines[["top","right"]].set_visible(False)
                ax_r.tick_params(labelsize=9)

    # x-axis label on bottom row
    for s in range(n_sides):
        axs[-1, s].set_xlabel("Time (s)", fontsize=10)

    mean_r2   = float(np.nanmean([d["metrics"][c]["r2"]   for c in range(N_OUT)]))
    mean_rmse = float(np.nanmean([d["metrics"][c]["rmse"] for c in range(N_OUT)]))
    _vs = "OpenSim ID − τ_exo" if _exo_on else "OpenSim ID"
    fig.suptitle(
        f"{stem}  |  GT IK → TCN → N·m vs {_vs} ({lpf_tag})  "
        f"Mean R² = {mean_r2:.3f}   Mean RMSE = {mean_rmse:.2f} N·m",
        fontsize=11, fontweight="bold",
    )

    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight", dpi=120)
    plt.close(fig)
    buf.seek(0)
    out.clear_output(wait=True)
    with out:
        display(Image(data=buf.read()))


# ── callbacks ─────────────────────────────────────────────────────────────────
def _on_trial(change):
    d = TRIAL_DATA[change["new"]]
    time_slider.min   = float(d["t_rel"][0])
    time_slider.max   = float(d["t_rel"][-1])
    time_slider.value = [float(d["t_rel"][0]), float(d["t_rel"][-1])]
    _redraw()

def _redraw(*_):
    _draw(trial_dropdown.value, *time_slider.value, show_residual.value)

trial_dropdown.observe(_on_trial,  names="value")
time_slider.observe(_redraw,        names="value")
show_residual.observe(_redraw,      names="value")

controls = widgets.HBox(
    [trial_dropdown, show_residual],
    layout=widgets.Layout(align_items="center", gap="12px"),
)
display(controls, time_slider, out)
_redraw()

FloatRangeSlider(value=(0.0, 79.99), continuous_update=False, description='Time (s):', layout=Layout(width='70…

Output()

## 7 · Window statistics

Re-run after moving the slider.

In [29]:
t0_s, t1_s = time_slider.value
stem_s     = trial_dropdown.value
d_s        = TRIAL_DATA[stem_s]
lpf_tag    = f"{ID_LPF_CUTOFF_HZ:.0f} Hz LPF"

mask_s   = (d_s["t_rel"] >= t0_s) & (d_s["t_rel"] <= t1_s)
pred_s   = d_s["pred_nm"][mask_s]   # (T_win, 6)
id_s_win = d_s["id_nm"][mask_s]     # (T_win, 6)

print(f"Trial  : {stem_s}")
print(f"Window : {t0_s:.1f} – {t1_s:.1f} s  ({t1_s-t0_s:.1f} s)")
print()
_id_hdr = "peak ID net" if d_s.get("exo_spec") else "peak ID"
print(f"{'Channel':<30} {'RMSE [N·m]':>10} {'R²':>7} {'peak model':>11} {_id_hdr:>9}")
print("-" * 72)
for c, (key, label) in enumerate(OUTPUT_LABELS):
    v = ~np.isnan(id_s_win[:, c])
    if v.sum() > 1:
        rmse_c = float(np.sqrt(np.mean((pred_s[v, c] - id_s_win[v, c])**2)))
        r2_c   = float(np.corrcoef(pred_s[v, c], id_s_win[v, c])[0, 1])**2
        pk_m   = float(np.nanmax(np.abs(pred_s[v, c])))
        pk_i   = float(np.nanmax(np.abs(id_s_win[v, c])))
    else:
        rmse_c = r2_c = pk_m = pk_i = np.nan
    print(f"  {label:<28} {rmse_c:>10.3f} {r2_c:>7.3f} {pk_m:>11.2f} {pk_i:>9.2f}")

mean_r2   = float(np.nanmean([float(np.corrcoef(pred_s[~np.isnan(id_s_win[:,c]),c],
                                                 id_s_win[~np.isnan(id_s_win[:,c]),c])[0,1])**2
                               for c in range(N_OUT) if (~np.isnan(id_s_win[:,c])).sum()>1]))
mean_rmse = float(np.nanmean([float(np.sqrt(np.mean((pred_s[~np.isnan(id_s_win[:,c]),c]
                                                      - id_s_win[~np.isnan(id_s_win[:,c]),c])**2)))
                               for c in range(N_OUT) if (~np.isnan(id_s_win[:,c])).sum()>1]))
print()
print(f"  {'Mean (all channels)':<28} {mean_rmse:>10.3f} {mean_r2:>7.3f}")

Trial  : awinda_LG_0p8mps
Window : 0.0 – 80.0 s  (80.0 s)

Channel                        RMSE [N·m]      R²  peak model peak ID net
------------------------------------------------------------------------
  Hip Flexion R                     8.440   0.833       42.71     32.76
  Knee Angle R                      3.801   0.867       23.45     25.38
  Ankle Angle R                    12.712   0.946      112.22    118.55
  Hip Flexion L                    15.458   0.783       52.37     32.24
  Knee Angle L                      7.121   0.789       27.79     36.01
  Ankle Angle L                    15.305   0.903      118.92    119.64

  Mean (all channels)              10.473   0.853


## 8 · All-trials summary (full overlap)

In [30]:
rows = []
for stem, d in TRIAL_DATA.items():
    row = {"trial": stem, "exo": d["exo"], "condition": d["cond"],
           f"ID LPF": f"{ID_LPF_CUTOFF_HZ:.0f} Hz {ID_LPF_ORDER}-ord causal",
           "GT ID": "ID − τ_exo" if d.get("exo_spec") else "ID gross",
           "overlap [s]": round(float(d["t_rel"][-1] - d["t_rel"][0]), 1)}
    for c, (key, label) in enumerate(OUTPUT_LABELS):
        m = d["metrics"][c]
        row[f"RMSE_{key} [N·m]"] = round(m["rmse"], 2) if not np.isnan(m["rmse"]) else np.nan
        row[f"R²_{key}"]          = round(m["r2"],   3) if not np.isnan(m["r2"])   else np.nan
    row["mean_RMSE [N·m]"] = round(float(np.nanmean([d["metrics"][c]["rmse"] for c in range(N_OUT)])), 2)
    row["mean_R²"]          = round(float(np.nanmean([d["metrics"][c]["r2"]   for c in range(N_OUT)])), 3)
    rows.append(row)

pd.DataFrame(rows)

,trial,exo,condition,ID LPF,GT ID,overlap [s],RMSE_hip_flexion_r [N·m],R²_hip_flexion_r,RMSE_knee_angle_r [N·m],R²_knee_angle_r,RMSE_ankle_angle_r [N·m],R²_ankle_angle_r,RMSE_hip_flexion_l [N·m],R²_hip_flexion_l,RMSE_knee_angle_l [N·m],R²_knee_angle_l,RMSE_ankle_angle_l [N·m],R²_ankle_angle_l,mean_RMSE [N·m],mean_R²
0,awinda_LG_0p8mps,awinda,LG_0p8mps,6 Hz 4-ord causal,ID − τ_exo,80.0,8.44,0.833,3.80,0.867,12.71,0.946,15.46,0.783,7.12,0.789,15.31,0.903,10.47,0.853
1,awinda_LG_1p2mps,awinda,LG_1p2mps,6 Hz 4-ord causal,ID − τ_exo,80.0,9.04,0.891,4.26,0.893,11.33,0.974,12.34,0.897,7.35,0.871,11.41,0.959,9.29,0.914
2,awinda_LG_1p6mps,awinda,LG_1p6mps,6 Hz 4-ord causal,ID − τ_exo,80.0,9.99,0.925,4.57,0.915,12.83,0.984,11.00,0.938,5.94,0.937,14.01,0.962,9.72,0.943
3,awinda_RA_0p8mps,awinda,RA_0p8mps,6 Hz 4-ord causal,ID − τ_exo,80.0,10.67,0.921,9.54,0.895,11.97,0.937,9.95,0.946,7.70,0.905,16.41,0.883,11.04,0.914
4,awinda_RD_0p8mps,awinda,RD_0p8mps,6 Hz 4-ord causal,ID − τ_exo,80.0,7.67,0.824,7.79,0.891,8.92,0.930,10.08,0.835,11.61,0.913,8.77,0.933,9.14,0.888
